
## Projeto: Merca Data Platform
Squad 2 | Helpers Centralizados



In [0]:

import os
import io
import time
import logging
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

from datetime import datetime
from dotenv import load_dotenv



In [0]:
ADLS_STORAGE_ACCOUNT = "internshipdatalake"
ADLS_CONTAINER = "real-time-ecommerce-data"

In [0]:

# ============================================================
# LOGS
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

log = logging.getLogger("squad2")

# ============================================================
# CREDENCIAIS
# ============================================================

load_dotenv()

# ADLS
ADLS_CLIENT_ID = os.getenv("ADLS_CLIENT_ID")
ADLS_TENANT_ID = os.getenv("ADLS_TENANT_ID")
ADLS_CLIENT_SECRET = os.getenv("ADLS_CLIENT_SECRET")

ADLS_STORAGE_ACCOUNT = os.getenv("STORAGE_ACCOUNT_NAME") or "internshipdatalake"
ADLS_CONTAINER = os.getenv("CONTAINER_NAME") or "real-time-ecommerce-data"

# SQL SERVER
SQL_HOST = os.getenv("SQL_HOST")
SQL_DATABASE = os.getenv("SQL_DATABASE")
SQL_USERNAME = os.getenv("SQL_USERNAME")
SQL_PASSWORD = os.getenv("SQL_PASSWORD")

# ============================================================
# CONFIGURAÇÕES DO PROJETO
# ============================================================

PATHS = {
    "raw": "vendas_raw",
    "bronze": "bronze/ecommerce_rastreamento",
    "silver": "silver/ecommerce_rastreamento",
    "gold": "gold/ecommerce_rastreamento",
    "checkpoint": "checkpoints/ecommerce_rastreamento"
}

TABELAS_SQUAD2 = [
    "ecommerce_rastreamento"
]

SQL_SCHEMA = "squad2"
SQL_PREFIX = ""


# ============================================================
# SNAPSHOTS
# ============================================================

def listar_snapshots():

    container_client = get_container_client()

    snapshots = set()

    for path in container_client.get_paths():

        partes = path.name.split("/")

        if (
            len(partes) >= 5
            and partes[0] == "vendas_raw"
        ):

            snapshot = "/".join(partes[1:5])

            snapshots.add(snapshot)

    return sorted(list(snapshots))


def get_snapshot_mais_recente():

    snapshots = listar_snapshots()

    if not snapshots:
        raise Exception(
            "Nenhum snapshot encontrado."
        )

    return sorted(snapshots)[-1]

# ============================================================
# CONFIGURAÇÃO ADLS
# ============================================================

from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient
def get_adls_client() -> DataLakeServiceClient:
    
    credential = ClientSecretCredential(
        tenant_id=ADLS_TENANT_ID,
        client_id=ADLS_CLIENT_ID,
        client_secret=ADLS_CLIENT_SECRET
    )

    return DataLakeServiceClient(
        account_url=f"https://{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net",
        credential=credential
    )


def get_container_client():
    """
    Retorna o cliente do container configurado.
    """
    return get_adls_client().get_file_system_client(ADLS_CONTAINER)


def configurar_adls():

    spark.conf.set(
        f"fs.azure.account.auth.type.{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net",
        "OAuth"
    )

    spark.conf.set(
        f"fs.azure.account.oauth.provider.type.{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net",
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
    )

    spark.conf.set(
        f"fs.azure.account.oauth2.client.id.{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net",
        ADLS_CLIENT_ID
    )

    spark.conf.set(
        f"fs.azure.account.oauth2.client.secret.{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net",
        ADLS_CLIENT_SECRET
    )

    spark.conf.set(
        f"fs.azure.account.oauth2.client.endpoint.{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net",
        f"https://login.microsoftonline.com/{ADLS_TENANT_ID}/oauth2/token"
    )

    log.info("ADLS configurado com sucesso.")

# ============================================================
# LEITURA DE ARQUIVOS
# ============================================================

def get_base_path():

    return (
        f"abfss://{ADLS_CONTAINER}@"
        f"{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net"
    )

def ler_parquet(path_relativo):

    path_completo = f"{get_base_path()}/{path_relativo}"

    return (
        spark.read
        .format("parquet")
        .load(path_completo)
    )

# ============================================================
# SQL SERVER
# ============================================================

SQL_OPTIONS = {
    "host": SQL_HOST,
    "database": SQL_DATABASE,
    "user": SQL_USERNAME,
    "password": SQL_PASSWORD
}

def get_destino_sql(tabela):

    return f"{SQL_SCHEMA}.{SQL_PREFIX}{tabela}"

def gravar_sql(df, tabela, mode="overwrite"):

    destino = get_destino_sql(tabela)

    (
        df.write
        .format("sqlserver")
        .options(**SQL_OPTIONS)
        .option("dbtable", destino)
        .mode(mode)
        .save()
    )

    log.info(f"Tabela gravada: {destino}")

def ler_sql(tabela):

    destino = get_destino_sql(tabela)

    return (
        spark.read
        .format("sqlserver")
        .options(**SQL_OPTIONS)
        .option("dbtable", destino)
        .load()
    )

# ============================================================
# LOGS
# ============================================================

def log_inicio(nome_notebook):

    inicio = datetime.now()

    log.info("=" * 50)
    log.info(f"INÍCIO: {nome_notebook}")
    log.info("=" * 50)

    return inicio

def log_fim(nome_notebook, inicio):

    fim = datetime.now()

    duracao = (fim - inicio).seconds

    log.info("=" * 50)
    log.info(f"FIM: {nome_notebook}")
    log.info(f"Duração: {duracao}s")
    log.info("=" * 50)

# ============================================================
# VALIDAÇÃO
# ============================================================

def validar_credenciais():

    credenciais = {
        "ADLS_CLIENT_ID": ADLS_CLIENT_ID,
        "ADLS_TENANT_ID": ADLS_TENANT_ID,
        "ADLS_CLIENT_SECRET": ADLS_CLIENT_SECRET,
        "ADLS_STORAGE_ACCOUNT": ADLS_STORAGE_ACCOUNT,
        "ADLS_CONTAINER": ADLS_CONTAINER,
        "SQL_HOST": SQL_HOST,
        "SQL_DATABASE": SQL_DATABASE,
        "SQL_USERNAME": SQL_USERNAME,
        "SQL_PASSWORD": SQL_PASSWORD
    }

    for nome, valor in credenciais.items():

        if not valor:
            raise ValueError(
                f"Credencial ausente: {nome}"
            )

    log.info("Helpers carregados com sucesso.")

validar_credenciais()